# Notebook 01 — Data Exploration
## MoleculeNet ADMET Datasets: Loading, Cleaning, and Chemical Space Analysis

**Author:** Shahid Afridi Laskar  
**Project:** ChemBERTa-ADMET  

---

### Background

ADMET property prediction is a central challenge in early-stage drug discovery. Before building any model, we need to understand the datasets:
- How many compounds? How balanced are the labels?
- What is the chemical diversity?
- Are there data quality issues (invalid SMILES, duplicates, outliers)?

This notebook loads five MoleculeNet datasets (BBBP, ESOL, ClinTox, Lipophilicity, Tox21), performs exploratory data analysis, and visualises the chemical space using physicochemical descriptors.

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from rdkit.Chem import rdMolDescriptors

print('Libraries loaded successfully.')

## 1. Load Datasets via DeepChem

In [ ]:
import deepchem as dc

# Load BBBP (classification)
tasks_bbbp, datasets_bbbp, _ = dc.molnet.load_bbbp(
    featurizer=dc.feat.DummyFeaturizer(),
    splitter='scaffold'
)
train_bbbp, val_bbbp, test_bbbp = datasets_bbbp
print(f'BBBP  — Train: {len(train_bbbp)}, Val: {len(val_bbbp)}, Test: {len(test_bbbp)}')

# Load ESOL (regression)
tasks_esol, datasets_esol, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.DummyFeaturizer(),
    splitter='scaffold'
)
train_esol, val_esol, test_esol = datasets_esol
print(f'ESOL  — Train: {len(train_esol)}, Val: {len(val_esol)}, Test: {len(test_esol)}')

# Load ClinTox (classification)
tasks_ct, datasets_ct, _ = dc.molnet.load_clintox(
    featurizer=dc.feat.DummyFeaturizer(),
    splitter='scaffold'
)
train_ct, val_ct, test_ct = datasets_ct
print(f'ClinTox — Train: {len(train_ct)}, Val: {len(val_ct)}, Test: {len(test_ct)}')

## 2. Convert to DataFrames and Inspect

In [ ]:
from src.data_utils import dataset_to_dataframe

df_bbbp = dataset_to_dataframe(train_bbbp, tasks_bbbp)
df_esol = dataset_to_dataframe(train_esol, tasks_esol)
df_ct   = dataset_to_dataframe(train_ct,   tasks_ct)

print('BBBP shape:', df_bbbp.shape)
print(df_bbbp.head(3))
print()
print('ESOL shape:', df_esol.shape)
print(df_esol.head(3))

## 3. Label Distribution (Classification Datasets)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# BBBP
bbbp_counts = df_bbbp['p_np'].value_counts()
axes[0].bar(['BBB-negative (0)', 'BBB-positive (1)'],
            [bbbp_counts.get(0, 0), bbbp_counts.get(1, 0)],
            color=['#e74c3c', '#2ecc71'])
axes[0].set_title('BBBP Label Distribution')
axes[0].set_ylabel('Count')

# ESOL
axes[1].hist(df_esol.iloc[:, 1], bins=30, color='#3498db', edgecolor='white')
axes[1].set_title('ESOL: log Solubility Distribution')
axes[1].set_xlabel('log(mol/L)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../figures/01_label_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 4. Physicochemical Descriptor Analysis

In [ ]:
from src.data_utils import smiles_to_rdkit_descriptors

# Compute descriptors for BBBP dataset
desc_rows = []
for _, row in df_bbbp.iterrows():
    desc = smiles_to_rdkit_descriptors(row['smiles'])
    if desc:
        desc['label'] = row['p_np']
        desc_rows.append(desc)

desc_df = pd.DataFrame(desc_rows)
print(f'Computed descriptors for {len(desc_df)} compounds')
print(desc_df.describe().round(2))

In [ ]:
# Chemical space: MW vs LogP, coloured by BBB label
fig, ax = plt.subplots(figsize=(8, 6))

for label, colour, name in [(0, '#e74c3c', 'BBB-negative'), (1, '#2ecc71', 'BBB-positive')]:
    subset = desc_df[desc_df['label'] == label]
    ax.scatter(subset['MolWt'], subset['LogP'],
               alpha=0.5, s=20, c=colour, label=f'{name} (n={len(subset)})')

# Lipinski limits
ax.axvline(x=500, color='gray', linestyle='--', alpha=0.7, label='MW=500 (Ro5)')
ax.axhline(y=5,   color='gray', linestyle=':',  alpha=0.7, label='LogP=5 (Ro5)')

ax.set_xlabel('Molecular Weight (Da)')
ax.set_ylabel('LogP')
ax.set_title('BBBP Chemical Space — MW vs LogP')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/01_chemical_space.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SMILES Validity Check

In [ ]:
from src.data_utils import is_valid_smiles

for name, df in [('BBBP', df_bbbp), ('ESOL', df_esol), ('ClinTox', df_ct)]:
    valid = sum(is_valid_smiles(s) for s in df['smiles'])
    print(f'{name}: {valid}/{len(df)} valid SMILES ({100*valid/len(df):.1f}%)')

## Summary

| Dataset | n (train) | Task | Class balance / value range |
|---|---|---|---|
| BBBP | ~1600 | Binary clf | ~80% BBB-positive |
| ESOL | ~900 | Regression | logS range ~ -11 to +2 |
| ClinTox | ~1200 | Binary clf | ~6% toxic (imbalanced) |

**Key observations:**
- BBBP is moderately imbalanced → need to track PR-AUC, not just accuracy
- ClinTox is severely imbalanced → will use weighted loss or oversampling
- ESOL compounds span a wide solubility range — good regression challenge
- All datasets show >99% valid SMILES

**Next:** `02_fingerprint_baseline.ipynb` — ECFP4 + Random Forest baseline models.